# Chain-of-Thought (CoT) Game of 24 Solver - OpenAI Version

Simple CoT approach (no tree search) using GPT-4o-mini.

**Approach:**
- Single-path reasoning (not beam search)
- Uses CoT prompt to guide step-by-step solution
- Outputs structured JSON with problem, steps, and answer

## 1. Setup - Import OpenAI Configuration

In [2]:
# Import OpenAI library
import openai
from openai import OpenAI
import os
import json
import re
from datetime import datetime
from typing import List, Dict, Any

print("✓ Libraries imported")

✓ Libraries imported


In [ ]:
# OpenAI Configuration (from tot_concept_openai_version.ipynb)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "") 

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY environment variable not set!")

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Model configuration
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.7
API_DELAY = 0.15  # seconds between calls

print(f"✓ OpenAI client initialized")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  API Delay: {API_DELAY}s")

✓ OpenAI client initialized
  Model: gpt-4o-mini
  Temperature: 0.7
  API Delay: 0.15s


## 2. CoT Prompt Definition

In [46]:
# Chain-of-Thought prompt for Game of 24
COT_PROMPT = '''Use numbers and basic arithmetic operations (+ - * /) to obtain 24. 

CRITICAL RULES:
1. When you combine two numbers, those two numbers are REMOVED and replaced by the result
2. Start with 4 numbers → after step 1: exactly 3 numbers → after step 2: exactly 2 numbers → after step 3: exactly 1 number
3. VERIFY YOUR ARITHMETIC: If you write "6 * 4 = 24", the left side must show 24, NOT the original 6 or 4
4. The "(left: ...)" shows what numbers remain AFTER this calculation

STRICT FORMAT - Follow these examples EXACTLY:
- Each step: "a op b = result (left: remaining_numbers)"
- DO NOT reuse consumed numbers in the "left" list
- After exactly 3 steps, you should have "(left: 24)" if successful
- Then write "Answer: expression" showing the full mathematical expression
- NO explanations, NO alternatives, NO commentary

Input: 4 4 6 8
Steps:
4 + 8 = 12 (left: 4 6 12)
6 - 4 = 2 (left: 2 12)
2 * 12 = 24 (left: 24)
Answer: (6 - 4) * (4 + 8)
Input: 2 9 10 12
Steps:
12 * 2 = 24 (left: 9 10 24)
10 - 9 = 1 (left: 1 24)
24 * 1 = 24 (left: 24)
Answer: (12 * 2) * (10 - 9)
Input: 4 9 10 13
Steps:
13 - 10 = 3 (left: 3 4 9)
9 - 3 = 6 (left: 4 6)
4 * 6 = 24 (left: 24)
Answer: 4 * (9 - (13 - 10))
Input: 1 4 8 8
Steps:
8 / 4 = 2 (left: 1 2 8)
1 + 2 = 3 (left: 3 8)
3 * 8 = 24 (left: 24)
Answer: (1 + 8 / 4) * 8
Input: 5 5 5 9
Steps:
5 + 5 = 10 (left: 5 9 10)
10 + 5 = 15 (left: 9 15)
15 + 9 = 24 (left: 24)
Answer: ((5 + 5) + 5) + 9
Input: {input}
Steps:
'''

SYSTEM_PROMPT = "You solve Game of 24 puzzles. Output ONLY steps and answer in the exact format shown. No explanations, no alternatives, no extra text."

print("✓ CoT prompt defined")
print(f"  Format: Step-by-step reasoning with intermediate states")
print(f"  Examples: 5 different puzzles shown")

✓ CoT prompt defined
  Format: Step-by-step reasoning with intermediate states
  Examples: 5 different puzzles shown


## 3. Helper Functions

In [47]:
import time

def call_openai(prompt: str, system_prompt: str = SYSTEM_PROMPT, temperature: float = TEMPERATURE) -> str:
    """
    Call OpenAI API with rate limiting
    
    Args:
        prompt: User prompt
        system_prompt: System instruction
        temperature: Sampling temperature
        
    Returns:
        Generated text response
    """
    time.sleep(API_DELAY)  # Rate limiting
    
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error calling OpenAI: {e}")
        return ""

print("✓ OpenAI helper function defined")

✓ OpenAI helper function defined


In [48]:
def verify_step_arithmetic(step: str) -> Dict[str, Any]:
    """
    Verify that the arithmetic in a step is correct
    
    Args:
        step: A step string like "6 * 4 = 24 (left: 2 24)"
        
    Returns:
        Dictionary with verification results
    """
    try:
        # Extract the calculation part (before "left:")
        calc_part = step.split('(left:')[0].strip()
        
        # Parse "a op b = result"
        parts = calc_part.split('=')
        if len(parts) != 2:
            return {'valid': False, 'error': 'Invalid format'}
        
        expression = parts[0].strip()
        claimed_result = parts[1].strip()
        
        # Try to evaluate the expression
        try:
            actual_result = eval(expression)
            # Convert to string for comparison (handle floats)
            if isinstance(actual_result, float) and actual_result.is_integer():
                actual_result = int(actual_result)
            
            actual_str = str(actual_result)
            is_correct = actual_str == claimed_result
            
            return {
                'valid': True,
                'is_correct': is_correct,
                'expression': expression,
                'claimed_result': claimed_result,
                'actual_result': actual_str,
                'error': None if is_correct else f"Wrong: {expression} = {actual_str}, not {claimed_result}"
            }
        except Exception as e:
            return {
                'valid': False,
                'is_correct': False,
                'error': f'Cannot evaluate: {str(e)}'
            }
    except Exception as e:
        return {
            'valid': False,
            'is_correct': False,
            'error': f'Parse error: {str(e)}'
        }

print("✓ Arithmetic verification function defined")

✓ Arithmetic verification function defined


In [52]:
def parse_cot_response(response: str) -> Dict[str, Any]:
    """
    Parse the CoT response to extract steps and final answer
    
    Args:
        response: Raw LLM response
        
    Returns:
        Dictionary with steps, final result, expression, and verification
    """
    lines = response.strip().split('\n')
    
    steps = []
    expression = ""
    final_result = None
    arithmetic_errors = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # Check if it's a step (contains '=' and 'left:')
        if '=' in line and 'left:' in line.lower():
            steps.append(line)
            
            # Verify arithmetic for this step
            verification = verify_step_arithmetic(line)
            if verification['valid'] and not verification['is_correct']:
                arithmetic_errors.append({
                    'step': line,
                    'error': verification['error']
                })
            
            # Extract the final number from the last step's (left: ...) part
            try:
                left_part = line.split('left:')[1].strip().rstrip(')')
                # Get the last number in the left part
                numbers = left_part.split()
                if numbers:
                    final_result = numbers[-1] if len(numbers) == 1 else ' '.join(numbers)
            except:
                pass
        # Check if it's the answer line (the expression)
        elif line.lower().startswith('answer:'):
            expression = line.replace('Answer:', '').replace('answer:', '').strip()
    
    # Check if solution is correct (final result is 24)
    is_solution = final_result == "24"
    has_arithmetic_errors = len(arithmetic_errors) > 0
    
    return {
        'steps': steps,
        'final_result': final_result,
        'expression': expression,
        'is_solution': is_solution,
        'has_arithmetic_errors': has_arithmetic_errors,
        'arithmetic_errors': arithmetic_errors
    }

print("✓ Response parser defined")

✓ Response parser defined


## 4. Main CoT Solver

In [53]:
def solve_game24_cot(numbers: List[int], verbose: bool = True) -> Dict[str, Any]:
    """
    Solve Game of 24 using Chain-of-Thought reasoning
    
    Args:
        numbers: List of 4 numbers
        verbose: Print progress
        
    Returns:
        Dictionary with problem, steps, answer, and metadata
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"Solving Game of 24: {numbers}")
        print(f"{'='*60}")
    
    # Format input for prompt
    input_str = ' '.join(map(str, numbers))
    prompt = COT_PROMPT.format(input=input_str)
    
    # Call LLM
    start_time = time.time()
    if verbose:
        print("\n🤖 Calling GPT-4o-mini...")
    
    response = call_openai(prompt)
    
    elapsed_time = time.time() - start_time
    
    if verbose:
        print(f"✓ Response received ({elapsed_time:.2f}s)\n")
        print("Raw response:")
        print("-" * 60)
        print(response)
        print("-" * 60)
    
    # Parse response
    parsed = parse_cot_response(response)
    
    # Build result JSON
    result = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "model": MODEL_NAME,
            "temperature": TEMPERATURE,
            "elapsed_time_seconds": round(elapsed_time, 2)
        },
        "problem": {
            "numbers": numbers,
            "input_string": input_str
        },
        "solution": {
            "steps": parsed['steps'],
            "final_result": parsed['final_result'],
            "expression": parsed['expression'],
            "is_solution": parsed['is_solution'],
            "has_arithmetic_errors": parsed['has_arithmetic_errors'],
            "arithmetic_errors": parsed['arithmetic_errors'],
            "raw_response": response
        }
    }
    
    if verbose:
        print("\n📊 Parsed Solution:")
        print(f"Steps ({len(parsed['steps'])})")
        for i, step in enumerate(parsed['steps'], 1):
            print(f"  {i}. {step}")
        
        # Show arithmetic errors if any
        if parsed['has_arithmetic_errors']:
            print(f"\n⚠️  ARITHMETIC ERRORS DETECTED:")
            for err in parsed['arithmetic_errors']:
                print(f"  - {err['error']}")
        
        print(f"\n📐 Expression: {parsed['expression']}")
        print(f"✅ Final Result: {parsed['final_result']}")
        
        if parsed['has_arithmetic_errors']:
            print(f"❌ INVALID (arithmetic errors)")
        elif parsed['is_solution']:
            print(f"✅ SOLVED!")
        else:
            print(f"❌ Not 24")
    
    return result

print("✓ CoT solver function defined")

✓ CoT solver function defined


## 5. Test the Solver

In [ ]:
# Test with example puzzle
test_numbers = [5, 6, 7, 13]

result = solve_game24_cot(test_numbers, verbose=True)

In [ ]:
# Display the JSON result
print("\n📄 Complete JSON Result:")
print("=" * 60)
print(json.dumps(result, indent=2))
print("=" * 60)

## 6. Save Result to JSON File

In [54]:
def save_result(result: Dict[str, Any], filename: str = None) -> str:
    """
    Save result to JSON file
    
    Args:
        result: Result dictionary
        filename: Optional custom filename
        
    Returns:
        Path to saved file
    """
    # Create output directory if it doesn't exist
    output_dir = "cot_openai"
    os.makedirs(output_dir, exist_ok=True)
    
    if filename is None:
        # Auto-generate filename with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        numbers_str = '_'.join(map(str, result['problem']['numbers']))
        filename = f"game24_cot_{numbers_str}_{timestamp}.json"
    
    # Add directory to filepath
    filepath = os.path.join(output_dir, filename)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Result saved to: {filepath}")
    return filepath

# Save the test result
# output_file = save_result(result)
# print(f"\n📁 File location: {os.path.abspath(output_file)}")

## 7. Batch Processing

In [55]:
def solve_batch(puzzle_list: List[List[int]], save_individual: bool = True) -> List[Dict[str, Any]]:
    """
    Solve multiple puzzles and save results
    
    Args:
        puzzle_list: List of number lists
        save_individual: Save each result to separate file
        
    Returns:
        List of result dictionaries
    """
    results = []
    
    for i, numbers in enumerate(puzzle_list, 1):
        print(f"\n\n{'='*60}")
        print(f"Puzzle {i}/{len(puzzle_list)}: {numbers}")
        print(f"{'='*60}")
        
        result = solve_game24_cot(numbers, verbose=True)
        results.append(result)
        
        if save_individual:
            save_result(result)
    
    # Save combined results
    output_dir = "cot_openai"
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    batch_file = os.path.join(output_dir, f"game24_cot_batch_{timestamp}.json")
    
    batch_data = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "total_puzzles": len(puzzle_list),
            "model": MODEL_NAME
        },
        "results": results
    }
    
    with open(batch_file, 'w', encoding='utf-8') as f:
        json.dump(batch_data, f, indent=2, ensure_ascii=False)
    
    print(f"\n\n{'='*60}")
    print(f"✅ Batch complete! {len(results)} puzzles solved")
    print(f"📁 Batch results saved to: {batch_file}")
    print(f"{'='*60}")
    
    return results

print("✓ Batch solver defined")

✓ Batch solver defined


In [56]:
# Example: Solve multiple puzzles
puzzles = [
    [4, 5, 6, 10],
    [1, 2, 4, 7],
    [2, 5, 8, 11],
    [3, 4, 4, 13],
    [6, 7, 8, 9],
    [1, 11, 11, 13],
    [1, 8, 10, 11],
    [2, 3, 6, 9],
    [1, 3, 5, 9],
    [3, 3, 7, 12],
    [4, 5, 7, 9],
    [1, 2, 8, 13],
    [4, 6, 6, 9],
    [1, 4, 4, 8],
    [1, 5, 10, 11],
    [3, 4, 6, 11],
    [2, 4, 8, 9],
    [1, 4, 5, 13],
    [2, 2, 7, 12],
    [3, 3, 6, 7],
    [1, 5, 9, 13],
    [5, 6, 7, 13]
]

print(f"📋 Loaded {len(puzzles)} puzzles for batch solving")
print("To run: batch_results = solve_batch(puzzles, save_individual=True)")

# Uncomment to run batch solving:
# batch_results = solve_batch(puzzles, save_individual=True)

📋 Loaded 22 puzzles for batch solving
To run: batch_results = solve_batch(puzzles, save_individual=True)


## 8. Summary

**Chain-of-Thought (CoT) Approach:**
- ✅ Simple single-path reasoning
- ✅ Fast (~1-2 seconds per puzzle)
- ✅ Low API usage (1 call per puzzle)
- ✅ Structured JSON output
- ⚠️ May not find solution on hard puzzles (no search/backtracking)

**Output Format:**
```json
{
  "metadata": { "timestamp", "model", "temperature", "elapsed_time" },
  "problem": { "numbers", "input_string" },
  "solution": { "steps", "answer", "raw_response" }
}
```

**Comparison with ToT:**
- CoT: 1 API call, 1-2s, simple
- ToT: 15-150 API calls, 2-10min, thorough

Use CoT for quick testing, ToT for guaranteed solutions on hard puzzles.

## 9. Run Batch Solver

Execute the batch solver on all 22 puzzles.

In [57]:
# Run batch solver on all 22 puzzles
# This will take approximately 30-45 seconds (22 puzzles × ~1.5s each)

print(f"Starting batch solve for {len(puzzles)} puzzles...")
print(f"Estimated time: ~{len(puzzles) * 1.5:.0f} seconds")
print("="*60)

batch_results = solve_batch(puzzles, save_individual=True)

Starting batch solve for 22 puzzles...
Estimated time: ~33 seconds


Puzzle 1/22: [4, 5, 6, 10]

Solving Game of 24: [4, 5, 6, 10]

🤖 Calling GPT-4o-mini...
✓ Response received (2.22s)

Raw response:
------------------------------------------------------------
10 - 6 = 4 (left: 4 5 4)  
4 + 4 = 8 (left: 5 8)  
5 * 8 = 40 (left: 40)  
Answer: (10 - 6) + (4 + 4) * 5
------------------------------------------------------------

📊 Parsed Solution:
Steps (3)
  1. 10 - 6 = 4 (left: 4 5 4)
  2. 4 + 4 = 8 (left: 5 8)
  3. 5 * 8 = 40 (left: 40)

📐 Expression: (10 - 6) + (4 + 4) * 5
✅ Final Result: 40
❌ Not 24
✓ Result saved to: cot_openai\game24_cot_4_5_6_10_20260204_104750.json


Puzzle 2/22: [1, 2, 4, 7]

Solving Game of 24: [1, 2, 4, 7]

🤖 Calling GPT-4o-mini...
✓ Response received (1.82s)

Raw response:
------------------------------------------------------------
4 * 2 = 8 (left: 1 7 8)  
7 - 1 = 6 (left: 6 8)  
6 * 8 = 48 (left: 48)  
Answer: (4 * 2) * (7 - 1)
-----------------------------

In [ ]:
# Display summary statistics
import pandas as pd

summary_data = []
for i, result in enumerate(batch_results, 1):
    numbers = result['problem']['numbers']
    final_result = result['solution']['final_result']
    expression = result['solution']['expression']
    is_solution = result['solution']['is_solution']
    has_errors = result['solution']['has_arithmetic_errors']
    num_steps = len(result['solution']['steps'])
    elapsed = result['metadata']['elapsed_time_seconds']
    
    # Determine status
    if has_errors:
        status = '⚠️'  # Arithmetic error
    elif is_solution:
        status = '✅'  # Correct
    else:
        status = '❌'  # Wrong result
    
    summary_data.append({
        '#': i,
        'Puzzle': str(numbers),
        'Steps': num_steps,
        'Result': final_result,
        'Status': status,
        'Expression': expression[:25] + '...' if len(expression) > 25 else expression,
        'Time(s)': elapsed
    })

summary_df = pd.DataFrame(summary_data)
print("\n📊 Batch Results Summary:")
print("="*80)
display(summary_df)

print("\n📈 Statistics:")
print(f"  Total puzzles: {len(batch_results)}")
solved_count = summary_df['Status'].value_counts().get('✅', 0)
error_count = summary_df['Status'].value_counts().get('⚠️', 0)
failed_count = summary_df['Status'].value_counts().get('❌', 0)
print(f"  ✅ Solved correctly: {solved_count}/{len(batch_results)}")
print(f"  ⚠️  Arithmetic errors: {error_count}/{len(batch_results)}")
print(f"  ❌ Failed (no errors): {failed_count}/{len(batch_results)}")
print(f"  Success rate: {(solved_count / len(batch_results) * 100):.1f}%")
print(f"  Average steps: {summary_df['Steps'].mean():.1f}")
print(f"  Average time: {summary_df['Time(s)'].mean():.2f}s")
print(f"  Total time: {summary_df['Time(s)'].sum():.1f}s")

NameError: name 'batch_results' is not defined